In [1]:
# --- [CELL 0]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 1}
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import warnings
warnings.filterwarnings('ignore') 

#!pip install sweetviz

import os
#import sweetviz as sv
import pickle
from sklearn_pandas import DataFrameMapper
from sklearn.preprocessing import OneHotEncoder,MinMaxScaler,StandardScaler,LabelEncoder
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.ensemble import GradientBoostingRegressor,RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

import matplotlib.pyplot as plt

In [2]:
# --- [CELL 1]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 2}
df = pd.read_csv('data/car data.csv')
df.head()

,Car_Name,Year,Selling_Price,Present_Price,Driven_kms,Fuel_Type,Selling_type,Transmission,Owner
0,ritz,2014,3.35,5.59,27000,Petrol,Dealer,Manual,0
1,sx4,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0
2,ciaz,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0
3,wagon r,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0
4,swift,2014,4.60,6.87,42450,Diesel,Dealer,Manual,0


In [3]:
# --- [CELL 2]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 3}
# 删除重复数据
df.drop_duplicates(inplace = True)
df.duplicated().sum()  # 检查是否已删除

0

In [4]:
# --- [CELL 3]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 4}
# 一种转换方法
dfm = DataFrameMapper([(['Year'],StandardScaler()),
                       (['Selling_Price'],None),
                       (['Driven_kms'],MinMaxScaler()),
                       ('Owner',None),
                       (['Car_Name'],OneHotEncoder()),
                       (['Fuel_Type'],OneHotEncoder()),
                       (['Selling_type'],OneHotEncoder()),
                       (['Transmission'],OneHotEncoder()),
                       (['Present_Price'],MinMaxScaler())
                      ],df_out=True)
transformed = dfm.fit_transform(df)

In [5]:
# --- [CELL 4]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 5}
# 第二种 不处理数值
# 文本类型的处理上不再使用独热编码
dfm2 = DataFrameMapper([(['Year'],None),
                        (['Selling_Price'],None),
                        (['Driven_kms'],None),
                        ('Owner',None),
                        (['Car_Name'],LabelEncoder()),
                        (['Fuel_Type'],LabelEncoder()),
                        (['Selling_type'],LabelEncoder()),
                        (['Transmission'],LabelEncoder()),
                        (['Present_Price'],None)
                       ],df_out=True)
transformed2 = dfm2.fit_transform(df)

In [6]:
# --- [CELL 5]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 6}
# 提取转换后的
X = transformed.loc[:,~transformed.columns.isin(['Selling_Price'])]
y = transformed.loc[:,'Selling_Price']

# 优化独热编码列名
def changename(name):
    for i in range(len(df[name].unique())):
        s = name + '_' + str(i)
        X.columns = [col.replace(s, name + '_' + df[name].unique()[i]) for col in X.columns]

changename('Selling_type')
changename('Transmission')
changename('Fuel_Type')
changename('Car_Name')

X.columns.tolist()

['Year',
 'Driven_kms',
 'Owner',
 'Car_Name_ritz',
 'Car_Name_sx4',
 'Car_Name_ciaz',
 'Car_Name_wagon r',
 'Car_Name_swift',
 'Car_Name_vitara brezza',
 'Car_Name_s cross',
 'Car_Name_alto 800',
 'Car_Name_ertiga',
 'Car_Name_dzire',
 'Car_Name_sx40',
 'Car_Name_sx41',
 'Car_Name_sx42',
 'Car_Name_sx43',
 'Car_Name_sx44',
 'Car_Name_sx45',
 'Car_Name_sx46',
 'Car_Name_sx47',
 'Car_Name_sx48',
 'Car_Name_sx49',
 'Car_Name_ciaz0',
 'Car_Name_ciaz1',
 'Car_Name_ciaz2',
 'Car_Name_ciaz3',
 'Car_Name_ciaz4',
 'Car_Name_ciaz5',
 'Car_Name_ciaz6',
 'Car_Name_ciaz7',
 'Car_Name_ciaz8',
 'Car_Name_ciaz9',
 'Car_Name_wagon r0',
 'Car_Name_wagon r1',
 'Car_Name_wagon r2',
 'Car_Name_wagon r3',
 'Car_Name_wagon r4',
 'Car_Name_wagon r5',
 'Car_Name_wagon r6',
 'Car_Name_wagon r7',
 'Car_Name_wagon r8',
 'Car_Name_wagon r9',
 'Car_Name_swift0',
 'Car_Name_swift1',
 'Car_Name_swift2',
 'Car_Name_swift3',
 'Car_Name_swift4',
 'Car_Name_swift5',
 'Car_Name_swift6',
 'Car_Name_swift7',
 'Car_Name_swi

In [7]:
# --- [CELL 6]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 7}
# 提取转换2的
X_ = transformed2.loc[:,~transformed2.columns.isin(['Selling_Price'])]
y_ = transformed2.loc[:,'Selling_Price']

In [8]:
# --- [CELL 7]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 8}
# 将数据集分割为训练集与测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=20)

X_2_train, X_2_test, y_2_train, y_2_test = train_test_split(X_, y_, test_size=0.1, random_state=20)

In [9]:
# --- [CELL 8]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 9}
# 默认方式
gbr = GradientBoostingRegressor(random_state=20)
gbr.fit(X_train,y_train)
gbr_y_predict = gbr.predict(X_test)

In [10]:
# --- [CELL 9]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 10}
gbr2 = GradientBoostingRegressor(random_state=20)
gbr2.fit(X_2_train,y_2_train)
gbr2_y_predict = gbr2.predict(X_2_test)

In [11]:
# --- [CELL 10]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 11}
gbr_gs = GradientBoostingRegressor(loss='squared_error', n_estimators=133,random_state=20)
gbr_gs.fit(X_2_train,y_2_train)
gbr_gs_y_predict = gbr_gs.predict(X_2_test)
gbr_gs.score(X_2_test,y_2_test)

0.9740787800578905

In [12]:
# --- [CELL 11]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 12}
#默认
rfr = RandomForestRegressor(random_state=20)
rfr.fit(X_train,y_train)
rfr_y_predict = rfr.predict(X_test)

In [13]:
# --- [CELL 12]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 13}
rfr2 = RandomForestRegressor(random_state=20)
rfr2.fit(X_2_train,y_2_train)
rfr2_y_predict = rfr2.predict(X_2_test)

In [14]:
# --- [CELL 13]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 14}
rfr_gs = RandomForestRegressor(criterion = 'friedman_mse', 
                               n_estimators = 136,
                               random_state=20)
rfr_gs.fit(X_2_train,y_2_train)
rfr_gs_y_predict = rfr_gs.predict(X_2_test)

In [15]:
# --- [CELL 14]: ---
# cell_state: edited
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 15}
# === BEFORE (original) ===
# # 表格方式
# model_dict = {'X_train': [gbr, gbr_gs, rfr, rfr_gs],
#               'X_2_train': [gbr2, rfr2]}
# 
# for key in model_dict:
#     f_list = []
#     for model in model_dict[key]:
#         feature_importance = model.feature_importances_
#         f_list.append(feature_importance)
#     
#     fearture_names = X_train.columns.tolist() if key == 'X_train' else X_2_train.columns.tolist()
#     f_index = ['gbr', 'gbr_gs', 'rfr', 'rfr_gs'] if key == 'X_train' else ['gbr2','rfr2']
#     
#     feature_df = pd.DataFrame(np.array(f_list), columns=fearture_names, index=f_index)
#     display(feature_df)

# === AFTER (edited) ===
model_dict = {'X_train': [gbr, gbr_gs, rfr, rfr_gs],
              'X_2_train': [gbr2, rfr2]}

for key, models in model_dict.items():
    feature_names = X_train.columns.tolist() if key == 'X_train' else X_2_train.columns.tolist()
    f_index = ['gbr', 'gbr_gs', 'rfr', 'rfr_gs'] if key == 'X_train' else ['gbr2', 'rfr2']

    feature_df = pd.DataFrame(
        [model.feature_importances_ for model in models],
        columns=feature_names,
        index=f_index
    )
    display(feature_df)

,Year,Driven_kms,Owner,Car_Name_ritz,Car_Name_sx4,Car_Name_ciaz,Car_Name_wagon r,Car_Name_swift,Car_Name_vitara brezza,Car_Name_s cross,...,Car_Name_dzire6,Car_Name_dzire7,Fuel_Type_Petrol,Fuel_Type_Diesel,Fuel_Type_CNG,Selling_type_Dealer,Selling_type_Individual,Transmission_Manual,Transmission_Automatic,Present_Price
gbr,0.091409,0.040699,2.312564e-07,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000004,0.000000,0.000008,0.009120,0.000046,0.000014,0.000003,0.000092,0.000151,0.850118
gbr_gs,0.092986,0.040657,8.247336e-05,0.013199,6.763571e-03,0.000000e+00,1.596677e-04,8.461528e-01,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rfr,0.075880,0.027206,6.001490e-04,0.000021,6.996870e-08,2.204153e-07,7.871038e-07,1.535167e-07,5.195565e-08,1.534220e-07,...,0.000050,0.000003,0.000014,0.001256,0.001191,0.002603,0.001817,0.003185,0.001941,0.867199
rfr_gs,0.080173,0.027290,4.856890e-04,0.021361,2.251158e-03,3.472337e-03,2.099563e-03,8.628673e-01,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,Year,Driven_kms,Owner,Car_Name,Fuel_Type,Selling_type,Transmission,Present_Price
gbr2,0.092985,0.040325,0.000083,0.013028,0.006743,0.000000,0.000156,0.846681
rfr2,0.077362,0.030770,0.000051,0.017538,0.002411,0.003102,0.004033,0.864733


In [16]:
# Verify model grouping matches feature spaces and importance vector lengths.
assert 'model_dict' in globals(), 'model_dict should be defined by the previous cell'

expected_names = {
    'X_train': ['gbr', 'rfr'],
    'X_2_train': ['gbr_gs', 'rfr_gs', 'gbr2', 'rfr2'],
}
assert set(model_dict.keys()) == set(expected_names.keys())

for key, names in expected_names.items():
    expected_models = [globals()[name] for name in names]
    actual_models = model_dict[key]

    assert len(actual_models) == len(expected_models), f"Unexpected model count for {key}"

    # Order-insensitive identity check
    assert {id(m) for m in actual_models} == {id(m) for m in expected_models}, (
        f"Wrong models mapped to {key}"
    )

expected_feature_counts = {
    'X_train': X_train.shape[1],
    'X_2_train': X_2_train.shape[1],
}
for key, models in model_dict.items():
    for m in models:
        assert len(m.feature_importances_) == expected_feature_counts[key], (
            f'Feature importance length mismatch for {key}'
        )

AssertionError: Unexpected model count for X_train